# Notebook 04: Hierarchical Clustering & Linkage Comparison
**Project**: Maternal-Child Health Profiles Across Countries Through Unsupervised Clustering

### Objectives:
1. Implement Agglomerative Hierarchical Clustering across Ward, Average, and Complete linkages.
2. Calculate Cophenetic Correlation Coefficients to evaluate tree distortion.
3. Compare hierarchical cluster cuts with K-Means using the Adjusted Rand Index (ARI).

In [ ]:
import sys
import os
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram
from sklearn.cluster import KMeans

import config
from src.data_loader import get_latest_cross_sectional_dataset
from src.preprocessing import HealthDataPreprocessor
from src.hierarchical import HierarchicalClusterer

### 1. Fit Hierarchical Linkages & Cophenetic Scores

In [ ]:
raw_data = get_latest_cross_sectional_dataset(min_year=2000)
preprocessor = HealthDataPreprocessor(max_missing_country_pct=35.0, imputation_strategy="median")
_, _, standardized = preprocessor.fit_transform_pipeline(raw_data, save_artifacts=False)

X = standardized[config.FEATURE_COLUMNS].values
hier = HierarchicalClusterer(metric="euclidean")
coph_scores = hier.fit_linkages(X)

print("Cophenetic Correlation Coefficients:")
for link, score in coph_scores.items():
    print(f"  {link.capitalize()}: {score:.4f}")

### 2. Ward Linkage Dendrogram

In [ ]:
plt.figure(figsize=(16, 7))
country_names = standardized["CountryName"].tolist()
dendrogram(
    hier.linkage_matrices["ward"],
    labels=country_names,
    leaf_rotation=90,
    leaf_font_size=8,
    color_threshold=hier.linkage_matrices["ward"][-3, 2]
)
plt.title("Hierarchical Dendrogram (Ward Linkage, Euclidean Distance)")
plt.xlabel("Countries")
plt.ylabel("Distance")
plt.tight_layout()
plt.show()

### 3. Comparison with K-Means (K=4)

In [ ]:
km = KMeans(n_clusters=4, random_state=config.RANDOM_SEED, n_init=config.N_INIT)
km_labels = km.fit_predict(X)

comparison_df = hier.compare_with_kmeans(X, km_labels, n_clusters=4)
print(comparison_df)